In [51]:
import os

os.environ["HADOOP_HOME"] = r"C:\Users\91840\OneDrive\Documents\hadoop-win-utils"
os.environ["PATH"] += os.pathsep + os.path.join(
    os.environ["HADOOP_HOME"],
    "bin"
)

Load the Raw Dataset

In [52]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SupportIQ-Profiling")
    .master("local[*]")
    .getOrCreate()
)

raw_path = "../data/raw/support_tickets_raw.csv"

tickets_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(raw_path)
)

Basic Dataset Information

In [53]:
print(
    "Number of rows:",
    tickets_df.count()
)

print(
    "Number of columns:",
    len(tickets_df.columns)
)



Number of rows: 1224000
Number of columns: 18


In [54]:
tickets_df.printSchema()

root
 |-- ticket_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- agent_id: string (nullable = true)
 |-- created_date: string (nullable = true)
 |-- closed_date: date (nullable = true)
 |-- priority: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- status: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- issue_description: string (nullable = true)
 |-- resolution: string (nullable = true)
 |-- customer_region: string (nullable = true)
 |-- first_response_minutes: integer (nullable = true)
 |-- resolution_hours: double (nullable = true)
 |-- customer_satisfaction: double (nullable = true)
 |-- escalation_flag: string (nullable = true)



In [55]:
tickets_df.show(
    5,
    truncate=False
)

+----------+-----------+--------+------------+-----------+--------+-----------+------------------+-------+-----------+-------+-------------------------------------------------+-------------------------------------------+---------------+----------------------+----------------+---------------------+---------------+
|ticket_id |customer_id|agent_id|created_date|closed_date|priority|category   |sub_category      |product|status     |channel|issue_description                                |resolution                                 |customer_region|first_response_minutes|resolution_hours|customer_satisfaction|escalation_flag|
+----------+-----------+--------+------------+-----------+--------+-----------+------------------+-------+-----------+-------+-------------------------------------------------+-------------------------------------------+---------------+----------------------+----------------+---------------------+---------------+
|TKT0000001|CUST28203  |AG0930  |2026-05-28  |2026-06-0

IDENTIFY — Duplicate Records

In [56]:
duplicate_count = (
    tickets_df.count()
    -
    tickets_df.dropDuplicates().count()
)

print(
    "Duplicate rows:",
    duplicate_count
)

Duplicate rows: 19908


MEASURE — Duplicate Percentage

In [57]:
total_rows = tickets_df.count()

unique_rows = tickets_df.dropDuplicates().count()

duplicate_rows = (
    total_rows - unique_rows
)

duplicate_percentage = (
    duplicate_rows / total_rows
) * 100

print(
    "Total rows:",
    total_rows
)

print(
    "Unique rows:",
    unique_rows
)

print(
    "Duplicate rows:",
    duplicate_rows
)

print(
    "Duplicate percentage:",
    round(
        duplicate_percentage,
        2
    ),
    "%"
)

Total rows: 1224000
Unique rows: 1204092
Duplicate rows: 19908
Duplicate percentage: 1.63 %


INSPECT — Duplicate Records

In [58]:
duplicate_groups = (
    tickets_df
    .groupBy(tickets_df.columns)
    .count()
    .filter("count > 1")
)

In [59]:
duplicate_groups.show(
    10,
    truncate=False
)

+----------+-----------+--------+------------+-----------+--------+---------------+------------------+-------+-----------+-------+-------------------------------------------------+----------------------------------------------------+---------------+----------------------+----------------+---------------------+---------------+-----+
|ticket_id |customer_id|agent_id|created_date|closed_date|priority|category       |sub_category      |product|status     |channel|issue_description                                |resolution                                          |customer_region|first_response_minutes|resolution_hours|customer_satisfaction|escalation_flag|count|
+----------+-----------+--------+------------+-----------+--------+---------------+------------------+-------+-----------+-------+-------------------------------------------------+----------------------------------------------------+---------------+----------------------+----------------+---------------------+---------------+-----

DUPLICATES — FIX

In [60]:
tickets_clean_df = (
    tickets_df
    .dropDuplicates()
)

DUPLICATES — VERIFY

In [61]:
total_after = tickets_clean_df.count()

print(
    "Rows after duplicate removal:",
    total_after
)

Rows after duplicate removal: 1204092


In [62]:
remaining_duplicates = (
    tickets_clean_df.count()
    -
    tickets_clean_df.dropDuplicates().count()
)

print(
    "Remaining duplicates:",
    remaining_duplicates
)

Remaining duplicates: 0


IDENTIFY — Missing Values

In [63]:
from pyspark.sql import functions as F

null_counts = tickets_df.select(
    [
        F.count(
            F.when(
                F.col(column).isNull(),
                column
            )
        ).alias(column)
        for column in tickets_df.columns
    ]
)

null_counts.show(
    truncate=False
)


+---------+-----------+--------+------------+-----------+--------+--------+------------+-------+------+-------+-----------------+----------+---------------+----------------------+----------------+---------------------+---------------+
|ticket_id|customer_id|agent_id|created_date|closed_date|priority|category|sub_category|product|status|channel|issue_description|resolution|customer_region|first_response_minutes|resolution_hours|customer_satisfaction|escalation_flag|
+---------+-----------+--------+------------+-----------+--------+--------+------------+-------+------+-------+-----------------+----------+---------------+----------------------+----------------+---------------------+---------------+
|0        |61177      |61225   |0           |0          |58207   |0       |0           |0      |0     |0      |0                |61158     |0              |0                     |0               |60026                |0              |
+---------+-----------+--------+------------+-----------+---

MEASURE — Null Percentage

In [64]:
total_rows = tickets_df.count()

null_percentage = tickets_df.select(
    [
        (
            F.count(
                F.when(
                    F.col(column).isNull(),
                    column
                )
            ) / total_rows * 100
        ).alias(column)
        for column in tickets_df.columns
    ]
)

null_percentage.show(
    truncate=False
)

+---------+----------------+-----------------+------------+-----------+----------------+--------+------------+-------+------+-------+-----------------+-----------------+---------------+----------------------+----------------+---------------------+---------------+
|ticket_id|customer_id     |agent_id         |created_date|closed_date|priority        |category|sub_category|product|status|channel|issue_description|resolution       |customer_region|first_response_minutes|resolution_hours|customer_satisfaction|escalation_flag|
+---------+----------------+-----------------+------------+-----------+----------------+--------+------------+-------+------+-------+-----------------+-----------------+---------------+----------------------+----------------+---------------------+---------------+
|0.0      |4.99812091503268|5.002042483660131|0.0         |0.0        |4.75547385620915|0.0     |0.0         |0.0    |0.0   |0.0    |0.0              |4.996568627450981|0.0            |0.0                   |

FIX — Fill Only the Appropriate Columns

In [65]:
tickets_clean_df = tickets_clean_df.fillna(
    {
        "priority": "Unknown",
        "resolution": "Not Provided"
    }
)

VERIFY — Check Priority

In [66]:
tickets_clean_df.filter(
    F.col("priority").isNull()
).count()

0

In [67]:
tickets_clean_df.select(
    "priority"
).distinct().show()

+----------------+
|        priority|
+----------------+
|        critical|
|            High|
|         medium |
|       Very High|
|           high |
|             Low|
|      Super High|
|         Unknown|
|          Medium|
|            High|
|            HIGH|
|            high|
|          URGENT|
|        CRITICAL|
|Unknown Priority|
|             low|
|        Critical|
+----------------+



VERIFY — Check Resolution

In [68]:
tickets_clean_df.filter(
    F.col("resolution").isNull()
).count()

0

In [69]:
tickets_clean_df.select(
    "resolution"
).distinct().show(
    truncate=False
)

+----------------------------------------------------+
|resolution                                          |
+----------------------------------------------------+
|User permissions were corrected                     |
|Issue was resolved after configuration update       |
|Data was corrected in the system                    |
|System behavior was verified and corrected          |
|Configuration was updated                           |
|Issue was escalated to the appropriate team         |
|Employee was provided with the required instructions|
|Knowledge article was provided                      |
|Not Provided                                        |
+----------------------------------------------------+



IDENTIFY — Invalid Priority

In [70]:
valid_priorities = [
    "Low",
    "Medium",
    "High",
    "Critical",
    "Unknown"
]

In [71]:
tickets_clean_df.select(
    "priority"
).groupBy(
    "priority"
).count().orderBy(
    F.desc("count")
).show(20, truncate=False)

+----------------+------+
|priority        |count |
+----------------+------+
|Medium          |272088|
|Critical        |271535|
|High            |271494|
|Low             |271352|
|Unknown         |57196 |
|URGENT          |5952  |
|Very High       |5947  |
|Unknown Priority|5936  |
|Super High      |5874  |
|CRITICAL        |4724  |
|high            |4629  |
| low            |4594  |
| High           |4593  |
|critical        |4590  |
|HIGH            |4563  |
|high            |4516  |
|medium          |4509  |
+----------------+------+



In [72]:
invalid_priority_df = (
    tickets_df
    .filter(
        ~F.col("priority").isin(
            valid_priorities
        )
    )
)

In [73]:
invalid_priority_df.select(
    "ticket_id",
    "priority"
).show(
    20,
    truncate=False
)

+----------+----------------+
|ticket_id |priority        |
+----------+----------------+
|TKT0000029|Unknown Priority|
|TKT0000043|Very High       |
|TKT0000044|high            |
|TKT0000066|medium          |
|TKT0000099|HIGH            |
|TKT0000101|critical        |
|TKT0000103|Unknown Priority|
|TKT0000116|Very High       |
|TKT0000139|high            |
|TKT0000152|Super High      |
|TKT0000155| High           |
|TKT0000161|Unknown Priority|
|TKT0000177|critical        |
|TKT0000179|Very High       |
|TKT0000229|Super High      |
|TKT0000274|Very High       |
|TKT0000302|HIGH            |
|TKT0000354|Unknown Priority|
|TKT0000361|Unknown Priority|
|TKT0000362|Very High       |
+----------+----------------+
only showing top 20 rows


MEASURE — Invalid Priority Count

In [74]:
invalid_priority_count = (
    invalid_priority_df.count()
)

print(
    "Invalid priority records:",
    invalid_priority_count
)

Invalid priority records: 60433


In [75]:
total_rows = tickets_df.count()

invalid_priority_percentage = (
    invalid_priority_count
    / total_rows
) * 100

print(
    "Invalid priority percentage:",
    round(
        invalid_priority_percentage,
        2
    ),
    "%"
)

Invalid priority percentage: 4.94 %


IDENTIFY — Inconsistent Priority

In [76]:
priority_check_df = tickets_df.withColumn(
    "normalized_priority",
    F.initcap(
        F.trim(
            F.col("priority")
        )
    )
)

#initcap() converts the first letter of each word to uppercase and the remaining letters to lowercase.

In [77]:
priority_check_df.filter(
    F.col("priority")
    !=
    F.col("normalized_priority")
).select(
    "priority",
    "normalized_priority"
).show(
    20,
    truncate=False
)

+--------+-------------------+
|priority|normalized_priority|
+--------+-------------------+
|high    |High               |
|medium  |Medium             |
|HIGH    |High               |
|critical|Critical           |
|high    |High               |
| High   |High               |
|critical|Critical           |
|HIGH    |High               |
|CRITICAL|Critical           |
|URGENT  |Urgent             |
|high    |High               |
|high    |High               |
|CRITICAL|Critical           |
|URGENT  |Urgent             |
|medium  |Medium             |
|critical|Critical           |
|critical|Critical           |
| High   |High               |
|URGENT  |Urgent             |
|high    |High               |
+--------+-------------------+
only showing top 20 rows


FIX — Standardize Priority Formatting

In [78]:
tickets_clean_df = tickets_clean_df.withColumn(
    "priority",
    F.initcap(
        F.trim(
            F.col("priority")
        )
    )
)

FIX — Replace Truly Invalid Values

In [79]:
tickets_clean_df = tickets_clean_df.withColumn(
    "priority",
    F.when(
        F.col("priority").isin(
            valid_priorities
        ),
        F.col("priority")
    ).otherwise(
        "Unknown"
    )
)

VERIFY — Check Distinct Values

In [80]:
tickets_clean_df.select(
    "priority"
).distinct().show(
    truncate=False
)

+--------+
|priority|
+--------+
|High    |
|Low     |
|Unknown |
|Medium  |
|Critical|
+--------+



VERIFY — Check for Invalid Values

In [81]:
invalid_priority_after = tickets_clean_df.filter(
    ~F.col("priority").isin(
        valid_priorities
    )
)

In [82]:
print(
    "Invalid priority records after cleaning:",
    invalid_priority_after.count()
)

Invalid priority records after cleaning: 0


IDENTIFY — Invalid Customer Satisfaction

In [83]:
invalid_csat_df = tickets_df.filter(
    (F.col("customer_satisfaction") < 1)
    |
    (F.col("customer_satisfaction") > 5)
)

In [84]:
invalid_csat_df.select(
    "ticket_id",
    "customer_satisfaction"
).show(
    20
)

+----------+---------------------+
| ticket_id|customer_satisfaction|
+----------+---------------------+
|TKT0000001|                  6.0|
|TKT0000046|                  0.0|
|TKT0000047|                  6.0|
|TKT0000051|                  0.0|
|TKT0000072|                  7.0|
|TKT0000084|                  7.0|
|TKT0000317|                 10.0|
|TKT0000443|                 10.0|
|TKT0000586|                 10.0|
|TKT0000588|                  0.0|
|TKT0000611|                  6.0|
|TKT0000638|                  7.0|
|TKT0000650|                  7.0|
|TKT0000721|                  7.0|
|TKT0000795|                  6.0|
|TKT0000883|                  0.0|
|TKT0000948|                  0.0|
|TKT0001006|                  7.0|
|TKT0001095|                  6.0|
|TKT0001135|                  0.0|
+----------+---------------------+
only showing top 20 rows


MEASURE — Invalid CSAT

In [85]:
invalid_csat_count = (
    invalid_csat_df.count()
)

print(
    "Invalid CSAT records:",
    invalid_csat_count
)

Invalid CSAT records: 24480


In [86]:
total_rows = tickets_df.count()

invalid_csat_percentage = (
    invalid_csat_count
    / total_rows
) * 100

print(
    "Invalid CSAT percentage:",
    round(
        invalid_csat_percentage,
        2
    ),
    "%"
)

Invalid CSAT percentage: 2.0 %


FIX — Convert invalid values to NULL

In [87]:
tickets_clean_df = tickets_clean_df.withColumn(
    "customer_satisfaction",
    F.when(
        (F.col("customer_satisfaction") >= 1)
        &
        (F.col("customer_satisfaction") <= 5),
        F.col("customer_satisfaction")
    ).otherwise(
        None
    )
)

VERIFY — Check that invalid values are gone

In [88]:
invalid_csat_after = tickets_clean_df.filter(
    (F.col("customer_satisfaction") < 1)
    |
    (F.col("customer_satisfaction") > 5)
)

print(
    "Invalid CSAT after cleaning:",
    invalid_csat_after.count()
)

Invalid CSAT after cleaning: 0


IDENTIFY — Invalid Dates

In [89]:
tickets_df.select(
    "created_date"
).show(
    20,
    truncate=False
)

+------------+
|created_date|
+------------+
|2026-05-28  |
|2025-09-01  |
|2026-07-02  |
|2025-10-10  |
|2025-05-06  |
|2025-09-14  |
|2026-04-23  |
|2025-04-15  |
|2025-07-27  |
|2025-01-10  |
|2025-05-05  |
|2026-06-19  |
|2025-10-12  |
|2026-02-21  |
|2026-08-20  |
|2026-05-02  |
|2026-05-06  |
|2025-05-15  |
|2025-12-13  |
|2026-05-06  |
+------------+
only showing top 20 rows


In [90]:
invalid_created_dates_df = tickets_df.filter(
    F.expr("try_cast(created_date AS DATE)").isNull()
    & F.col("created_date").isNotNull()
)

invalid_created_dates_df.select(
    "ticket_id",
    "created_date"
).show(truncate=False)

+----------+------------+
|ticket_id |created_date|
+----------+------------+
|TKT0000024|not_a_date  |
|TKT0000055|2025-99-99  |
|TKT0000228|2026-02-30  |
|TKT0000377|2025-99-99  |
|TKT0000386|2026-02-31  |
|TKT0000433|2026-13-01  |
|TKT0000510|2026-02-30  |
|TKT0000589|2026-02-31  |
|TKT0000657|2026-13-01  |
|TKT0000677|2026-02-30  |
|TKT0001080|2026-02-31  |
|TKT0001092|not_a_date  |
|TKT0001098|2025-99-99  |
|TKT0001182|2025-99-99  |
|TKT0001280|2026-13-01  |
|TKT0001404|2025-99-99  |
|TKT0001515|not_a_date  |
|TKT0001611|2025-99-99  |
|TKT0001654|2026-02-31  |
|TKT0001724|2026-02-30  |
+----------+------------+
only showing top 20 rows


In [91]:
date_check_df = tickets_df.withColumn(
    "parsed_created_date",
    F.expr("try_cast(created_date AS DATE)")
)

In [92]:
invalid_dates_df = date_check_df.filter(
    F.col("created_date").isNotNull()
    &
    F.col("parsed_created_date").isNull()
)

In [93]:
invalid_dates_df.select(
    "ticket_id",
    "created_date"
).show(
    20,
    truncate=False
)

+----------+------------+
|ticket_id |created_date|
+----------+------------+
|TKT0000024|not_a_date  |
|TKT0000055|2025-99-99  |
|TKT0000228|2026-02-30  |
|TKT0000377|2025-99-99  |
|TKT0000386|2026-02-31  |
|TKT0000433|2026-13-01  |
|TKT0000510|2026-02-30  |
|TKT0000589|2026-02-31  |
|TKT0000657|2026-13-01  |
|TKT0000677|2026-02-30  |
|TKT0001080|2026-02-31  |
|TKT0001092|not_a_date  |
|TKT0001098|2025-99-99  |
|TKT0001182|2025-99-99  |
|TKT0001280|2026-13-01  |
|TKT0001404|2025-99-99  |
|TKT0001515|not_a_date  |
|TKT0001611|2025-99-99  |
|TKT0001654|2026-02-31  |
|TKT0001724|2026-02-30  |
+----------+------------+
only showing top 20 rows


MEASURE — Invalid Dates

In [94]:
invalid_date_count = (
    invalid_dates_df.count()
)

print(
    "Invalid date records:",
    invalid_date_count
)

Invalid date records: 12240


In [95]:
total_rows = tickets_df.count()

invalid_date_percentage = (
    invalid_date_count
    / total_rows
) * 100

print(
    "Invalid date percentage:",
    round(
        invalid_date_percentage,
        2
    ),
    "%"
)

Invalid date percentage: 1.0 %


FIX — Convert the column to a proper date

In [96]:
tickets_clean_df = tickets_clean_df.withColumn(
    "created_date",
    F.expr("try_cast(created_date AS DATE)")
)

VERIFY — Check invalid dates again

In [97]:
tickets_clean_df.select(
    "ticket_id",
    "created_date"
).show(
    20,
    truncate=False
)

+----------+------------+
|ticket_id |created_date|
+----------+------------+
|TKT0000141|2025-02-06  |
|TKT0000404|2025-01-10  |
|TKT0000479|2026-01-21  |
|TKT0000603|2025-08-30  |
|TKT0000984|2025-11-18  |
|TKT0001265|2025-04-24  |
|TKT0001548|2025-07-19  |
|TKT0001974|2025-09-14  |
|TKT0001982|2025-06-06  |
|TKT0001992|2025-08-22  |
|TKT0002177|2025-01-23  |
|TKT0002182|2025-08-04  |
|TKT0002424|2025-03-12  |
|TKT0002525|2025-04-17  |
|TKT0002610|2025-07-24  |
|TKT0002915|2026-08-24  |
|TKT0002970|2026-03-22  |
|TKT0002987|2025-07-27  |
|TKT0003376|2026-01-05  |
|TKT0003599|2025-02-07  |
+----------+------------+
only showing top 20 rows


In [98]:
tickets_clean_df.printSchema()

root
 |-- ticket_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- agent_id: string (nullable = true)
 |-- created_date: date (nullable = true)
 |-- closed_date: date (nullable = true)
 |-- priority: string (nullable = false)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- status: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- issue_description: string (nullable = true)
 |-- resolution: string (nullable = false)
 |-- customer_region: string (nullable = true)
 |-- first_response_minutes: integer (nullable = true)
 |-- resolution_hours: double (nullable = true)
 |-- customer_satisfaction: double (nullable = true)
 |-- escalation_flag: string (nullable = true)



IDENTIFY — Extreme Resolution Hours

In [99]:
outlier_df = tickets_df.filter(
    F.col("resolution_hours") > 500
)

In [100]:
outlier_df.select(
    "ticket_id",
    "resolution_hours"
).show(
    20
)

+----------+------------------+
| ticket_id|  resolution_hours|
+----------+------------------+
|TKT0000106| 23885.65728756857|
|TKT0000157|24254.376712540234|
|TKT0000188|32876.027447195025|
|TKT0000243| 5861.097259395708|
|TKT0000423|26354.314561521132|
|TKT0000435|30304.886076790775|
|TKT0000618| 6468.659914247177|
|TKT0000688|24337.443089411554|
|TKT0000723|20418.403609963374|
|TKT0000757|38704.797564603105|
|TKT0000786|  44459.6699357771|
|TKT0000835|25288.053313667868|
|TKT0000993|13449.487706899512|
|TKT0001341|27702.673286365793|
|TKT0001454|37000.549885930304|
|TKT0001485|26947.705853151732|
|TKT0001537|37995.075301092955|
|TKT0001736| 40910.07994012802|
|TKT0002045|34737.696369547855|
|TKT0002262| 22987.31528649764|
+----------+------------------+
only showing top 20 rows


MEASURE — Resolution Outliers

In [101]:
outlier_count = (
    outlier_df.count()
)

print(
    "Extreme resolution records:",
    outlier_count
)

Extreme resolution records: 12240


In [102]:
total_rows = tickets_df.count()

outlier_percentage = (
    outlier_count
    / total_rows
) * 100

print(
    "Extreme resolution percentage:",
    round(
        outlier_percentage,
        2
    ),
    "%"
)

Extreme resolution percentage: 1.0 %


FIX — Convert outliers to NULL

In [103]:
tickets_clean_df = tickets_clean_df.withColumn(
    "resolution_hours",
    F.when(
        F.col("resolution_hours") <= 500,
        F.col("resolution_hours")
    ).otherwise(
        None
    )
)

VERIFY — Check that outliers are gone

In [104]:
remaining_outliers = tickets_clean_df.filter(
    F.col("resolution_hours") > 500
)

print(
    "Outliers after cleaning:",
    remaining_outliers.count()
)

Outliers after cleaning: 0


Final Validation

In [105]:

print("       SUPPORTIQ DATA VALIDATION")

print(
    "Final rows:",
    tickets_clean_df.count()
)

print(
    "Remaining duplicates:",
    remaining_duplicates
)

print(
    "Invalid priority records:",
    invalid_priority_after.count()
)

print(
    "Invalid CSAT records:",
    invalid_csat_after.count()
)

print(
    "Remaining resolution outliers:",
    remaining_outliers.count()
)


       SUPPORTIQ DATA VALIDATION
Final rows: 1204092
Remaining duplicates: 0
Invalid priority records: 0
Invalid CSAT records: 0
Remaining resolution outliers: 0


Save Clean Data

Create the processed-data folder

In [106]:
import os

processed_path = "../data/processed"

os.makedirs(
    processed_path,
    exist_ok=True
)

print(
    "Processed folder ready:",
    processed_path
)

Processed folder ready: ../data/processed


Choose the output path

In [107]:
clean_path = "../data/processed/support_tickets_cleaned"

Save the cleaned DataFrame

In [108]:
tickets_clean_df.write \
    .mode("overwrite") \
    .parquet(clean_path)

Verify that the data was saved

In [111]:
cleaned_df = spark.read.parquet(
    clean_path
)

print(
    "Cleaned data loaded successfully."
)

Cleaned data loaded successfully.
